# Context-aware VLM highlight scores for FPVSum (Track 04, last dev experiment)

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, Persistence **Files only**, add the FPVSum dataset. **Run All** (~2–3 h).
Output: `/kaggle/working/vlm_ctx_scores.zip`.

Following zero-shot summarisers that score scenes *with whole-video context* (Prompts-to-Summaries, arXiv 2506.10807;
context-aware pseudo-label scoring, arXiv 2510.17501), instead of rating each clip in isolation:

1. **Caption** every non-overlapping 4-second window (4 frames @ 1 fps) with Qwen2.5-VL-7B.
2. **Summarise** the whole video from its timestamped captions (same model, text only).
3. **Score** each window *relative to the rest of the video*: the prompt holds the video summary, the previous and next captions and a rubric;
   score = expected digit under the model's next-token distribution. Two rubrics: highlight-reel value (p0) and distinctiveness (p1).
4. **Rank** (p2): the model reads the whole numbered caption list and names the segments that belong in a highlight reel, best first.

No ground truth, file names or categories are used. Resumable: re-running skips finished videos.

In [ ]:
!pip install -q "transformers==4.51.3" "accelerate>=0.34"
import torch, transformers
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import glob, os, time, zipfile, math, json, re
import numpy as np, cv2, torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

VIDEOS = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
CAP_DIR = "/kaggle/working/captions"; OUT = "/kaggle/working/vlm_ctx_scores"
os.makedirs(CAP_DIR, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(len(VIDEOS), "videos found"); assert VIDEOS, "attach the FPVSum dataset"
WIN = 4                      # 4-second windows, non-overlapping, 1 frame per second
SHORT, LONG = 252, 448
CAP_BATCH, SCORE_BATCH = 8, 8

MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
mm = {i: "13GiB" for i in range(torch.cuda.device_count())}
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto",
                                                           max_memory=mm, attn_implementation="sdpa").eval()
proc = AutoProcessor.from_pretrained(MODEL); proc.tokenizer.padding_side = "left"
DIG = [proc.tokenizer.encode(str(d), add_special_tokens=False)[0] for d in range(10)]

def to_dev(inp):
    return {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inp.items()}

def chat(content):
    return proc.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(texts, videos=None, max_new=60):
    kw = dict(videos=videos, fps=1.0) if videos is not None else {}
    inp = to_dev(proc(text=texts, padding=True, return_tensors="pt", **kw))
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return [s.strip() for s in proc.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)]

@torch.no_grad()
def digit_scores(texts):
    inp = to_dev(proc(text=texts, padding=True, return_tensors="pt"))
    p_all = torch.softmax(model(**inp).logits[:, -1, :].float(), -1)
    p = p_all[:, DIG]; mass = p.sum(-1); p = p / mass.clamp_min(1e-9)[:, None]
    return (p * torch.arange(10, device=p.device)).sum(-1).cpu().numpy(), mass.cpu().numpy()

print("loaded", MODEL)

In [ ]:
def frames_1fps(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); dur = n / fps
    want = [int(round((k + 0.5) * fps)) for k in range(int(math.floor(dur)))]
    out, i, j = [], 0, 0
    while j < len(want):
        if not cap.grab():
            break
        if i == want[j]:
            ok, f = cap.retrieve()
            if ok:
                h, w = f.shape[:2]; size = (LONG, SHORT) if w >= h else (SHORT, LONG)
                out.append(Image.fromarray(cv2.cvtColor(cv2.resize(f, size, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)))
            j += 1
        i += 1
    cap.release()
    return out, dur

CAP_PROMPT = ("These are 4 consecutive seconds from a first-person (head- or body-mounted camera) video. "
              "In one short sentence (at most 20 words) say what happens: the action, any notable event "
              "(jump, trick, crash, goal, landing, close encounter, big view) and the setting.")

# ---------------- stage 1: captions
t0 = time.time()
for vi, path in enumerate(VIDEOS):
    vid = os.path.splitext(os.path.basename(path))[0]; dst = os.path.join(CAP_DIR, vid + ".json")
    if os.path.exists(dst):
        continue
    frames, dur = frames_1fps(path)
    if len(frames) < WIN:
        frames = frames + [frames[-1]] * (WIN - len(frames))
    starts = list(range(0, len(frames) - WIN + 1, WIN))
    if starts[-1] + WIN < len(frames):
        starts.append(len(frames) - WIN)
    clips = [frames[s:s + WIN] for s in starts]
    text = chat([{"type": "video"}, {"type": "text", "text": CAP_PROMPT}])
    caps = []
    for b in range(0, len(clips), CAP_BATCH):
        caps += generate([text] * len(clips[b:b + CAP_BATCH]), clips[b:b + CAP_BATCH], max_new=40)
    json.dump(dict(t=[s + WIN / 2 for s in starts], captions=caps, duration=dur), open(dst, "w"))
    print(f"[captions] {vi + 1}/{len(VIDEOS)} {vid}: {len(caps)} windows  [{(time.time() - t0) / 60:.1f} min]  e.g. {caps[len(caps) // 2]!r}", flush=True)

In [ ]:
RUBRIC_HL = ("A video editor must cut a highlight reel that keeps only about 15% of this video. "
             "Judge THIS segment relative to the rest of this same video. Rubric: 9 = one of the very best moments of this video "
             "(peak action, climax, a unique event); 6-8 = clearly more interesting than most of the video; "
             "3-5 = typical footage of the activity; 0-2 = setup, waiting, repetitive or uneventful. Answer with one digit.")
RUBRIC_UNIQ = ("How distinctive is THIS segment within this video: does something happen here that does not happen elsewhere "
               "(a specific event, a change of scene, a peak or a surprise), or does it look like the rest of the footage? "
               "Rubric: 9 = clearly unique and memorable; 5 = somewhat different; 0 = indistinguishable from the rest. Answer with one digit.")

def fmt(t, c):
    return f"[{int(t - WIN / 2)}s] {c}"

def summary_prompt(lines):
    return chat([{"type": "text", "text": "Timestamped descriptions of consecutive 4-second segments of one first-person video:\n"
                  + "\n".join(lines) + "\n\nIn 3-4 sentences summarise the whole video: the activity, how it progresses, "
                  "and which moments stand out as the most exciting, important or unusual."}])

def score_prompt(summary, prev, cur, nxt, rubric):
    return chat([{"type": "text", "text": f"Summary of the whole video: {summary}\n\nPrevious segment: {prev}\nTHIS segment: {cur}\n"
                  f"Next segment: {nxt}\n\n{rubric}"}])

def rank_prompt(lines, k):
    return chat([{"type": "text", "text": "Numbered descriptions of consecutive 4-second segments of one first-person video:\n"
                  + "\n".join(f"{i}. {l}" for i, l in enumerate(lines)) +
                  f"\n\nChoose the {k} segments that most deserve to be in a short highlight reel of this video "
                  "(the most exciting, important or unique moments). Reply only with their numbers, best first, separated by commas."}])

# ---------------- stage 2: summary + context-aware scores + listwise ranking
t0 = time.time()
for vi, path in enumerate(VIDEOS):
    vid = os.path.splitext(os.path.basename(path))[0]; dst = os.path.join(OUT, vid + ".npz")
    if os.path.exists(dst):
        continue
    c = json.load(open(os.path.join(CAP_DIR, vid + ".json"))); t, caps = np.array(c["t"]), c["captions"]; n = len(caps)
    lines = [fmt(a, b) for a, b in zip(t, caps)]
    summary = generate([summary_prompt(lines)], max_new=200)[0]
    S = np.zeros((n, 3), np.float32); M = np.zeros((n, 2), np.float32)
    for q, rubric in enumerate([RUBRIC_HL, RUBRIC_UNIQ]):
        prompts = [score_prompt(summary, lines[i - 1] if i else "(start of video)", lines[i],
                                lines[i + 1] if i + 1 < n else "(end of video)", rubric) for i in range(n)]
        for b in range(0, n, SCORE_BATCH):
            s, m = digit_scores(prompts[b:b + SCORE_BATCH]); S[b:b + len(s), q] = s; M[b:b + len(s), q] = m
    k = max(2, int(round(0.25 * n)))
    reply = generate([rank_prompt(lines, k)], max_new=6 * k + 10)[0]
    picks = []
    for x in re.findall(r"\d+", reply):
        x = int(x)
        if 0 <= x < n and x not in picks:
            picks.append(x)
    for r, x in enumerate(picks[:k]):
        S[x, 2] = (k - r) / k
    np.savez_compressed(dst, t=t.astype(np.float32), score=S, mass=M, captions=np.array(caps, dtype=object),
                        summary=summary, rank_reply=reply, duration=c["duration"], model=MODEL)
    print(f"[scores] {vi + 1}/{len(VIDEOS)} {vid}: p0 {S[:, 0].mean():.2f}±{S[:, 0].std():.2f}  p1 {S[:, 1].mean():.2f}±{S[:, 1].std():.2f}  "
          f"ranked {len(picks)}/{k}  mass {M.mean():.2f}  [{(time.time() - t0) / 60:.1f} min]", flush=True)

In [ ]:
with zipfile.ZipFile("/kaggle/working/vlm_ctx_scores.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*.npz"))):
        z.write(f, "vlm_ctx_scores/" + os.path.basename(f))
print(len(glob.glob(os.path.join(OUT, "*.npz"))), "videos -> /kaggle/working/vlm_ctx_scores.zip")